[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/18_compound_ai_evaluation/53_compound_ai_evaluation_cafe.ipynb)

# 📓 Notebook 53 — Compound AI Evaluation with CAFE

> **Module:** Compound AI Evaluation · **Estimated time:** 75–90 min · **Difficulty:** Intermediate–Advanced

In NB 32 you built the smoke detector: golden datasets, LLM-as-judge, tracing. It tells you **that** quality moved. This notebook answers the harder question: **which part of the pipeline moved it — and is the difference real?**

Modern AI applications are **compound systems** — pipelines of interacting techniques: retrieval, reranking, context assembly, prompting, one or more model calls, tools, verifiers. When the output improves, *which knob actually helped?* An aggregate benchmark score can't say. The discipline that can is nearly a century old: **design of experiments (DoE)** — the same factorial designs and mixed-effects models that agronomy and clinical trials run on, pointed at an AI pipeline.

We cover:

- **The attribution problem** — why "we swapped the model *and* added a reranker, and it got better" is not a measurement.
- **Factors, levels, and factorial designs** — turning every pipeline knob into an experimental variable.
- **Replication** — separating real effects from LLM run-to-run noise.
- **Statistical attribution** — ANOVA F-tests, p-values, and effect sizes (partial η²) per factor, built by hand with `statsmodels`.
- **Cost/quality Pareto frontiers** — finding the configs where you can't get better without paying more.
- **[CAFE](https://github.com/fabian-lu/Cafe)** (*Compound-AI Factorial Evaluation*, [cafe-ai.de](https://cafe-ai.de)) — the open-source library that packages this whole workflow: factorial designs, an LLM-judge layer, scale-correct mixed-effects models, and a self-hostable web platform.

> 🎯 **Our running example: Meridian's support-answer bot.** The fictional B2B SaaS company from Module 16 has shipped a QA bot for its product *Meridian Flow*. The team just "improved" it — bigger model, added retrieval, rewrote the prompt, all in one release — and quality went up. The CFO asks: *the bigger model costs 5× — did it actually contribute anything, or was it the retrieval?* Nobody knows. By the end of this notebook, you will.

> 📎 **Everything up to the final section runs 100% offline** — we build the whole factorial-evaluation workflow by hand on a mock pipeline, exactly the course pattern from NB 27–32. The last section shows the same study expressed in a few lines of CAFE, which you can optionally install (it needs Python ≥ 3.11 and R).

*Terminology: "compound AI systems" (the term from the Berkeley AI Research blog that stuck) are also called "composite AI systems" — same thing: systems whose behaviour emerges from multiple interacting components rather than one model call.*

## 1. The attribution problem — "it got better" is not a measurement

Here's the release that triggered the CFO's question. Three changes shipped at once:

| Change | Cost impact |
|---|---|
| `mock-small` → `mock-large` model | **5× per call** |
| Added keyword retrieval over the docs | +ops complexity |
| Rewrote the prompt from terse to "grounded" | +input tokens |

Average judge score went from ≈0.8 to ≈2.8 on the golden set. Great — but which change did it? The possibilities are genuinely different business decisions:

- If it was mostly **retrieval**, roll the model back to `mock-small` and cut inference cost 5×.
- If it was mostly the **model**, the retrieval stack is dead weight — delete it.
- If it was the **combination** (retrieval only pays off when the model is big enough to use the context), you need both — and you should know that's an *interaction*, not two separate effects.

The failure mode is universal enough to have a name in experimental design: a **confounded comparison**. You changed several factors at once, so their effects are entangled in a single before/after number. The fix is not more vibes, it's a **factorial experiment**: vary each knob systematically, run every combination, and let statistics allocate the credit.

> 💡 **Why not just A/B test one change at a time?** One-factor-at-a-time (OFAT) is better than nothing, but it (a) can't see interactions, and (b) wastes runs — a factorial design reuses every run to estimate *every* factor simultaneously. This is R. A. Fisher's 1926 insight, and it applies verbatim to RAG pipelines.

## 2. Factors, levels, configurations — the vocabulary

Three terms carry the whole field:

- A **factor** is a pipeline knob you want to attribute quality to: *which retrieval strategy? which model? which prompt style?*
- Each factor has **levels** — the concrete options: `retrieval ∈ {none, keyword}`, `model_size ∈ {small, large}`, `prompt_style ∈ {terse, grounded}`.
- A **configuration** is one full assignment of levels — one runnable version of your system. A **full factorial design** runs *every* combination.

With 3 binary factors that's 2 × 2 × 2 = **8 configurations**. Enumerate them with `itertools.product` — this is the entire "design generation" step:

In [ ]:
import itertools
import random
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)

FACTORS = {
    "retrieval": ["none", "keyword"],
    "model_size": ["small", "large"],
    "prompt_style": ["terse", "grounded"],
}

configs = [dict(zip(FACTORS, combo)) for combo in itertools.product(*FACTORS.values())]
pd.DataFrame(configs)

Eight rows — eight versions of the bot we will actually run. The design *is* the experiment plan: nothing hidden, no sampling cleverness yet (that comes in §8 with fractional designs, for when the grid explodes).

---

### ✋ Quick exercise (~2 min) — How fast does the grid grow?

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

The team wants to test a fourth knob: `reranker ∈ {off, on}`. Add it to a **copy** of `FACTORS` and compute (a) the number of configurations, and (b) the total number of pipeline runs for a study with 10 questions and 6 replications per configuration.

In [ ]:
# ✍️ Your turn 👇
# Copy FACTORS, add "reranker": ["off", "on"], then compute both counts.
n_configs = ...
n_runs = ...
print(n_configs, n_runs)

<details>
<summary>✅ <b>Solution</b></summary>

```python
factors4 = {**FACTORS, "reranker": ["off", "on"]}
n_configs = len(list(itertools.product(*factors4.values())))
n_runs = n_configs * 10 * 6
print(n_configs, n_runs)   # 16 configs, 960 runs
```

Every added binary factor **doubles** the grid — 16 configs, 960 judged pipeline runs. That exponential growth is exactly why fractional factorial designs (§8) exist.
</details>

## 3. The system under test — a mock compound pipeline

CAFE's core design decision is that the system under test is a **black box**: any function with the shape `run(config, item) → output`. It doesn't need to see inside your pipeline — it only needs to be able to run a *configuration* on an *item*.

We'll honour the same contract. Our offline stand-in has the three real stages of a RAG bot — retrieve, assemble prompt, call model — with the mock model's correctness probabilities rigged the way real pipelines behave: **the right context helps a lot, model size helps some, and the grounded prompt only helps when there's context to ground on** (an interaction, hidden in the code — the experiment should rediscover it).

First, the golden set (the NB 32 artefact) and the product docs:

In [ ]:
# Meridian Flow's documentation — one fact per topic. This is what retrieval searches.
KB = {
    "export":    "Dashboards export from Settings → Share → Export as PDF.",
    "sso":       "Single sign-on is configured under Admin → Security → SSO with SAML 2.0.",
    "api_limit": "The public API allows 600 requests per minute per workspace.",
    "refund":    "Annual plans can be refunded within 30 days of purchase.",
    "import":    "CSV imports support files up to 50 MB via Data → Import.",
    "retention": "Audit logs are retained for 365 days on the Enterprise tier.",
    "webhook":   "Webhooks retry failed deliveries 5 times with exponential backoff.",
    "sandbox":   "Every workspace includes one free sandbox environment.",
    "billing":   "Invoices can be issued in EUR, USD, or GBP.",
    "mobile":    "The mobile app supports offline mode on iOS and Android.",
}

# The golden set: question + the reference phrase a correct answer must contain.
GOLDEN = [
    ("export",    "How do I export a dashboard as a PDF?",           "Settings → Share → Export"),
    ("sso",       "How do we enable single sign-on for our team?",   "Admin → Security → SSO"),
    ("api_limit", "What is the API rate limit?",                     "600 requests per minute"),
    ("refund",    "Can I get a refund on my annual plan?",           "within 30 days"),
    ("import",    "How large can a CSV import be?",                  "50 MB"),
    ("retention", "How long are audit logs kept?",                   "365 days"),
    ("webhook",   "What happens when a webhook delivery fails?",     "5 times"),
    ("sandbox",   "Do we get a test environment to try things in?",  "sandbox environment"),
    ("billing",   "Which currencies can we be billed in?",           "EUR, USD, or GBP"),
    ("mobile",    "Does the app work without an internet connection?", "offline mode"),
]
golden_df = pd.DataFrame(GOLDEN, columns=["topic", "question", "reference"])
print(f"{len(KB)} docs, {len(golden_df)} golden questions")
golden_df.head(3)

Now the pipeline stages. Note that keyword retrieval is *genuinely imperfect* — it finds the right doc for 8 of the 10 questions. Q7 asks what happens "when a **webhook** delivery fails", but the doc says "**Webhooks** retry…" — plural, and with no stemming the keywords don't overlap; the billing question ("which **currencies**…") shares no content word at all with its doc ("**Invoices** can be issued in EUR…"). Real retrieval fails in exactly these ways (vocabulary mismatch is *the* classic keyword-search failure — it's why NB 29 moved to embeddings), and the experiment prices the failures in automatically.

In [ ]:
STOPWORDS = {"how", "do", "i", "a", "the", "we", "our", "is", "are", "can", "what",
             "which", "for", "of", "be", "does", "get", "my", "on", "in", "to", "it",
             "without", "an", "when", "things", "try"}

def keywords(text):
    return {w for w in re.findall(r"[a-z]+", text.lower()) if w not in STOPWORDS}

def retrieve(mode, question):
    """Stage 1 — fetch a doc for the question (or don't)."""
    if mode == "none":
        return None
    overlap = [(len(keywords(question) & keywords(doc)), topic) for topic, doc in KB.items()]
    score, topic = max(overlap)
    return topic if score > 0 else None          # topic of the best-matching doc

def assemble_prompt(style, question, doc_topic):
    """Stage 2 — terse just asks; grounded quotes the retrieved doc and demands citation."""
    context = f"Context: {KB[doc_topic]}\n" if doc_topic else ""
    if style == "terse":
        return f"{context}Q: {question}"
    return (f"You are Meridian Flow support. Answer ONLY from the context; "
            f"quote the exact setting or number.\n{context}Q: {question}")

def mock_model(size, style, item_topic, doc_topic, rng):
    """Stage 3 — offline stand-in for the LLM call (the NB 27–32 MockLLM pattern).

    Returns a correct, vague, or wrong answer with probabilities that encode
    how real models behave: right context >> model size > prompt style."""
    has_right_doc = doc_topic == item_topic
    if has_right_doc:
        p_correct = {"small": 0.82, "large": 0.93}[size]
        if style == "grounded":                   # grounding only helps WITH context
            p_correct = min(p_correct + 0.05, 0.98)
    else:
        p_correct = {"small": 0.25, "large": 0.48}[size]   # parametric knowledge only

    roll = rng.random()
    if roll < p_correct:
        return f"According to the docs: {KB[item_topic]}"
    if roll < p_correct + 0.30:
        return "That's covered in the docs — check the relevant settings page."
    wrong = rng.choice([t for t in KB if t != item_topic])
    return f"According to the docs: {KB[wrong]}"

# Per-run cost in "credit units" — in production you'd log real tokens/latency per call.
MODEL_COST = {"small": 0.2, "large": 1.0}
RETRIEVAL_COST = {"none": 0.0, "keyword": 0.05}
PROMPT_COST = {"terse": 0.0, "grounded": 0.1}

def run_pipeline(config, item, rng):
    """The black box CAFE asks for: run(config, item) -> output."""
    topic, question, reference = item
    doc_topic = retrieve(config["retrieval"], question)
    _prompt = assemble_prompt(config["prompt_style"], question, doc_topic)
    answer = mock_model(config["model_size"], config["prompt_style"], topic, doc_topic, rng)
    cost = MODEL_COST[config["model_size"]] + RETRIEVAL_COST[config["retrieval"]] \
           + PROMPT_COST[config["prompt_style"]]
    return {"answer": answer, "cost": cost}

rng = random.Random(42)
demo = run_pipeline(configs[-1], GOLDEN[0], rng)      # best config, first question
print(demo["answer"], f"\n(cost: {demo['cost']} credits)")

> 🔌 **Using a real provider.** Swap `mock_model` for a call through [`llm_providers.py`](../llm_providers.py) (OpenAI / Anthropic / Gemini / local Ollama — see the [providers guide](../08_ai_engineering/A1_llm_providers_guide.ipynb)) and the *entire rest of this notebook works unchanged*. That's the point of the black-box contract: the experimental machinery never looks inside.

## 4. The judge — scoring answers on a rubric

Each answer gets a score on an explicit **rubric**. We use a 0–3 correctness scale — the same shape as CAFE's built-in `CORRECTNESS_0_3`:

| Score | Meaning |
|---|---|
| **3** | Correct — contains the reference answer |
| **2** | Correct but incomplete |
| **1** | Vague / non-answer, but not wrong |
| **0** | Wrong or misleading |

In NB 32 you built an **LLM-as-judge** for exactly this job. Here we keep a keyless keyword judge so everything runs offline (it never awards a 2 — a real LLM judge uses the full scale):

In [ ]:
def keyword_judge(answer, reference):
    """Offline judge: 3 = contains the reference, 1 = vague deflection, 0 = wrong."""
    if reference.lower() in answer.lower():
        return 3
    if "check the relevant settings page" in answer.lower():
        return 1
    return 0

print(keyword_judge(demo["answer"], GOLDEN[0][2]))    # the demo answer from §3

Two judge notes that matter in practice:

- **An ordinal rubric is not a number line.** The distance from 2→3 is not promised to equal 0→1. §6 comes back to why that changes the statistics.
- **Judges must themselves be validated.** The standard move is to have humans rate a sample of the same answers and measure judge↔human agreement with **Krippendorff's α** (an inter-rater reliability coefficient that handles ordinal scales and missing ratings; ≳0.8 is usually considered trustworthy). CAFE has human rating and α built in — in your own harness, even a simple exact-agreement rate on 50 double-rated answers beats trusting the judge blindly.

## 5. Replication — running the study

One run per configuration is how teams fool themselves. LLM calls are **nondeterministic** (temperature, provider-side changes, retrieval ties), so a single run of config A beating config B is one coin flip, not a conclusion. The fix is **replication**: run every (configuration × question) pair several times and let the noise show itself.

Full study = 8 configs × 10 questions × 6 replications = **480 judged runs**:

In [ ]:
REPLICATIONS = 6
rng = random.Random(0)

rows = []
for cfg in configs:
    for qid, item in enumerate(GOLDEN):
        for rep in range(REPLICATIONS):
            out = run_pipeline(cfg, item, rng)
            rows.append({**cfg, "qid": f"q{qid:02d}", "rep": rep,
                         "score": keyword_judge(out["answer"], item[2]),
                         "cost": out["cost"]})

results = pd.DataFrame(rows)
print(f"{len(results)} judged runs")
results.head()

Every row is one judged pipeline run — the tidy "long format" that every statistics tool wants. First look: mean score per configuration.

In [ ]:
CFG_COLS = ["retrieval", "model_size", "prompt_style"]

config_means = (results.groupby(CFG_COLS)["score"].agg(["mean", "std"])
                .sort_values("mean", ascending=False).round(2))
config_means

In [ ]:
ax = config_means["mean"].plot.barh(xerr=config_means["std"], color="#4c72b0")
ax.set_xlabel("mean judge score (0–3)")
ax.set_title("Mean quality per configuration (error bars: ±1 SD across runs)")
ax.invert_yaxis()
plt.tight_layout()

A clear gradient — and notice the standard deviations: they're **large relative to the gaps between neighbouring configs**. That's the LLM-noise problem in one picture. Watch what happens if we'd only run each config once (replication 0):

In [ ]:
winner_all = results.groupby(CFG_COLS)["score"].mean().idxmax()
winners_by_rep = (results.groupby("rep")
                  .apply(lambda d: d.groupby(CFG_COLS)["score"].mean().idxmax(),
                         include_groups=False))
print("Winner using ALL replications :", winner_all)
print("\nWinner if you had run only one replication:")
print(winners_by_rep.to_string())
print(f"\n{winners_by_rep.nunique()} different configs claim the crown across "
      f"{REPLICATIONS} single-replication studies.")

Different single-replication studies crown different winners — same pipeline, same golden set, same judge. Any one of them would have been written up as "config X is best" in a slide deck. Replication + significance testing (next section) is what separates a real effect from having gotten lucky.

---

### ✋ Quick exercise (~2 min) — How unstable is a single-replication study?

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

The winner is one number — look at the whole ranking instead. Using `results`, compute the mean score of the **all-replication winner config** (`winner_all`) *within each single replication*, and print the min and max. How far does one config's "measured quality" swing depending on which single study you ran?

In [ ]:
# ✍️ Your turn 👇
# Filter results to the winner_all config (hint: CFG_COLS, winner_all is a tuple),
# then compute its mean score per rep, and print min / max.
winner_rows = ...
per_rep = ...
print(per_rep)

<details>
<summary>✅ <b>Solution</b></summary>

```python
mask = (results[CFG_COLS] == pd.Series(winner_all, index=CFG_COLS)).all(axis=1)
winner_rows = results[mask]
per_rep = winner_rows.groupby("rep")["score"].mean()
print(per_rep)
print(f"swing: {per_rep.min():.2f} → {per_rep.max():.2f}")
```

The *same* configuration scores noticeably differently depending on which replication you happened to run — often by more than the gap to the runner-up. Any comparison that ignores this noise is reading tea leaves.
</details>

## 6. Attribution — which factor moves quality, and by how much?

Now the payoff. The config table says *which config* wins; the CFO asked *which change matters*. That's a question about **factors**, and the workhorse answer is the **analysis of variance (ANOVA)**: model the score as a function of the factors, then test how much variance each factor explains.

Two modelling details are load-bearing:

1. **Questions differ in difficulty**, and every config saw the same 10 questions. Adding `qid` to the model soaks up that per-question variance so it doesn't drown the factor effects — the classic *blocking* / repeated-measures move. (The fully honest version treats the question as a **random effect** in a mixed-effects model — that's what CAFE fits; we show both.)
2. We include the `retrieval × model_size` **interaction** term, because §1's whole worry was "maybe the model only pays off together with retrieval".

In [ ]:
ols = smf.ols("score ~ C(retrieval) * C(model_size) + C(prompt_style) + C(qid)",
              data=results).fit()
anova = sm.stats.anova_lm(ols, typ=2)

# Partial eta-squared: the share of (factor + residual) variance the factor explains.
anova["partial_eta_sq"] = anova["sum_sq"] / (anova["sum_sq"] + anova.loc["Residual", "sum_sq"])
anova.round(4)

How to read this table — the three columns that matter:

- **`PR(>F)`** (the p-value): *is the effect real?* Small (≲0.05) ⇒ the factor's effect is distinguishable from replication noise.
- **`partial_eta_sq`**: *how big is it?* The share of variance the factor explains, holding the others fixed — this is the ranking the CFO wants. (Rules of thumb: ~0.01 small, ~0.06 medium, ≥0.14 large.)
- **The interaction row** (`C(retrieval):C(model_size)`): if significant, "does the big model help?" has no single answer — it depends on retrieval, and you must report the combination.

And the effect *direction and size* in score units, straight from the group means:

In [ ]:
for f in CFG_COLS:
    levels = results.groupby(f)["score"].mean()
    print(f"{f:>13}: " + "  vs  ".join(f"{lv}={m:.2f}" for lv, m in levels.items())
          + f"   (Δ = {levels.max() - levels.min():+.2f})")

print("\nretrieval × model_size cell means:")
print(results.pivot_table(index="retrieval", columns="model_size",
                          values="score", aggfunc="mean").round(2))

The verdict for the CFO, in plain language: **retrieval is the star** (largest effect by far — partial η² several times everything else), **model size contributes a real but much smaller lift**, and the prompt rewrite is a small-but-real, nearly-free improvement. The significant **interaction** adds the subtle part: read the cell means — the large model's lift is big *without* retrieval (it falls back on parametric knowledge) and much smaller *with* it. The 5× model partially **substitutes** for retrieval rather than stacking on top of it — so once retrieval ships, the expensive model buys far less than the before/after release suggested.

### The mixed-effects upgrade (what CAFE actually fits)

Treating `qid` as 9 fixed dummy coefficients works, but the statistically-preferred model treats questions as a **random sample from a population of possible questions** — a *random intercept per question*. Same idea, better generalisation claim: the conclusion is about the technique, not these 10 questions.

In [ ]:
mixed = smf.mixedlm("score ~ retrieval * model_size + prompt_style",
                    data=results, groups=results["qid"]).fit()
print(mixed.summary().tables[1])

Same story, now with the per-question variance explicitly modelled (`Group Var`).

> ⚠️ **The ordinal caveat.** Both models above treat the 0–3 rubric as if 1→2 and 2→3 were equal steps. For a quick offline analysis that's a reasonable approximation — but it's an assumption, not a fact. The scale-correct model for an ordinal rubric is a **cumulative-link mixed model (CLMM)**, and for a binary pass/fail rubric a **logistic** mixed model. This is precisely CAFE's "scale-matched models" feature: it picks the correct family for your rubric automatically (running the CLMM through R's `ordinal` package). When your OLS p-value sits near the decision boundary, the scale-correct model is the one to trust.

---

### ✋ Quick exercise (~2 min) — Read the ANOVA like the CFO

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

From the `anova` table (already computed): print the factor rows sorted by `partial_eta_sq` (largest first), and answer with code: is the `retrieval × model_size` interaction significant at α = 0.05?

In [ ]:
# ✍️ Your turn 👇
# Sort anova by partial_eta_sq (drop the Residual row), then check the
# interaction row's PR(>F) against 0.05.
ranked = ...
interaction_significant = ...
print(ranked)
print("interaction significant:", interaction_significant)

<details>
<summary>✅ <b>Solution</b></summary>

```python
ranked = anova.drop(index="Residual").sort_values("partial_eta_sq", ascending=False)
interaction_significant = anova.loc["C(retrieval):C(model_size)", "PR(>F)"] < 0.05
print(ranked[["PR(>F)", "partial_eta_sq"]])
print("interaction significant:", interaction_significant)
```

Retrieval tops the ranking by a wide margin. The interaction comes out **significant** (p ≈ 0.003 with this seed): the big model's lift depends on retrieval — much larger when retrieval is off than on (see the cell means above). So "do we need the big model?" has no single answer; the honest report gives the combination.
</details>

## 7. Cost/quality trade-offs — the Pareto frontier

Quality attribution is half the CFO's question; the other half is money. Every run logged its cost, so we can place all 8 configurations on a quality-vs-cost plane. A config is **Pareto-optimal** if no other config is *both* cheaper *and* better — the set of Pareto-optimal configs forms the **frontier**, and every config off the frontier is strictly dominated: you could pay less and get more.

In [ ]:
summary = (results.groupby(CFG_COLS, as_index=False)
           .agg(quality=("score", "mean"), cost=("cost", "mean")))

def pareto_frontier(df, cost_col="cost", quality_col="quality"):
    """Rows not dominated by any cheaper-and-better row (assumes minimise cost, maximise quality)."""
    frontier, best = [], -np.inf
    for _, row in df.sort_values([cost_col, quality_col], ascending=[True, False]).iterrows():
        if row[quality_col] > best:
            frontier.append(row)
            best = row[quality_col]
    return pd.DataFrame(frontier)

frontier = pareto_frontier(summary)
frontier.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(summary["cost"], summary["quality"], s=60, color="#999999", label="dominated")
ax.scatter(frontier["cost"], frontier["quality"], s=90, color="#4c72b0", zorder=3,
           label="Pareto frontier")
ax.step(frontier["cost"], frontier["quality"], where="post", color="#4c72b0", alpha=0.5)
for _, r in summary.iterrows():
    ax.annotate(f"{r.retrieval[:4]}/{r.model_size}/{r.prompt_style[:5]}",
                (r.cost, r.quality), textcoords="offset points", xytext=(6, -3), fontsize=8)
ax.set_xlabel("cost per call (credits)")
ax.set_ylabel("mean judge score (0–3)")
ax.set_title("Quality vs cost — every configuration, frontier highlighted")
ax.legend()
plt.tight_layout()

This one chart is the whole business conversation. Typical readings:

- The **cheap frontier point** (keyword retrieval + small model) often delivers most of the quality at a fraction of the cost — that's the "roll back the model, keep retrieval" option, now with evidence.
- The **top-right frontier point** is "best quality money can buy" — the question is whether the last fraction of a point is worth the cost multiple.
- Anything **off the frontier** is simply a mistake — a config nobody should run. Note where `none/large` lands: paying 5× for the model *without* retrieval is usually dominated. That's the CFO's answer.

CAFE computes this frontier automatically over quality vs. cost, latency, or tokens.

---

### ✋ Quick exercise (~2 min) — The price drop

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

The provider halves the large model's price: `1.0 → 0.5` credits. Rebuild `summary` with the new cost (recompute a `cost2` column on `results` from its factor columns — don't re-run the pipelines!) and print the new frontier. Does `none/large` make it onto the frontier now?

In [ ]:
# ✍️ Your turn 👇
new_model_cost = {"small": 0.2, "large": 0.5}
# Recompute cost per row from retrieval/model_size/prompt_style, rebuild summary, re-derive frontier.
summary2 = ...
frontier2 = ...
print(frontier2)

<details>
<summary>✅ <b>Solution</b></summary>

```python
new_model_cost = {"small": 0.2, "large": 0.5}
cost2 = (results["model_size"].map(new_model_cost)
         + results["retrieval"].map(RETRIEVAL_COST)
         + results["prompt_style"].map(PROMPT_COST))
summary2 = (results.assign(cost=cost2).groupby(CFG_COLS, as_index=False)
            .agg(quality=("score", "mean"), cost=("cost", "mean")))
frontier2 = pareto_frontier(summary2)
print(frontier2.round(2))
```

Costs are a *property of the configuration*, so a price change never requires re-running the experiment — just re-deriving the frontier. (Quality stays fixed; only the x-axis moves.) `none/large` stays dominated: cheaper large-model calls don't fix the missing context — `keyword/large` gets better quality for only 0.05 more.
</details>

## 8. Doing it for real — CAFE

Everything above was ~80 lines of hand-rolled harness — great for understanding, but a real study needs more: crash-safe concurrent execution over hundreds of runs, a managed LLM-judge layer, the *scale-correct* CLMM/logistic models (via R), human-rating workflows with Krippendorff's α, fractional designs, and a UI your team can read. That's what **[CAFE](https://github.com/fabian-lu/Cafe)** packages:

| We hand-rolled | CAFE concept |
|---|---|
| `FACTORS` dict + `itertools.product` | `factors=[...]` → full/**fractional** factorial designs |
| `run_pipeline(config, item, rng)` | the black-box `run(config, item)` contract, or `@pipe.technique` / `@pipe.compose` decorators |
| `keyword_judge` | `cafe.LLMJudge(model=...)` + human raters, agreement via Krippendorff's α |
| 0–3 rubric table | `cafe.rubrics.CORRECTNESS_0_3` (or your own) |
| `for rep in range(6)` loop | `replications=6` — concurrent, crash-safe, resumable |
| OLS/MixedLM + partial η² | scale-matched mixed-effects models (ordinal → **CLMM**, binary → logistic, in R) with F-tests, p-values, partial η² |
| `pareto_frontier()` | automatic Pareto frontier over quality vs cost/latency/tokens |
| this notebook 🙂 | a self-hostable FastAPI + React platform with live progress ([demo](https://cafe-ai.de/demo)) |

**Install** (needs Python ≥ 3.11 **and R** — the CLMM/GLMM models run through R's `ordinal`/`lme4`):

```bash
git clone https://github.com/fabian-lu/Cafe.git && cd Cafe
pip install -e "packages/cafe-core[all]"
# macOS: brew install r        Debian/Ubuntu: sudo apt install r-base
Rscript -e 'install.packages(c("ordinal", "lme4"))'
cafe doctor          # verify Python + R + LLM access
cafe run example     # bundled toy study — no API keys needed
```

Here is our §2–§6 study expressed in CAFE — same shape, a fraction of the code. The cell below only runs if you've installed `cafe-core` (it's the library's keyless quick-start pattern: toy techniques, toy judge, zero API keys):

In [ ]:
try:
    import cafe
    CAFE_AVAILABLE = True
except ImportError:
    CAFE_AVAILABLE = False
    print("cafe-core not installed — cell below becomes a no-op. "
          "Install it (plus R) to run the real thing; the code is worth reading either way.")

In [ ]:
if CAFE_AVAILABLE:
    pipe = cafe.Pipeline()

    @pipe.technique("retrieval", "none")
    async def no_retrieval(ctx, query):
        return None

    @pipe.technique("retrieval", "keyword")
    async def kw_retrieval(ctx, query):
        return retrieve("keyword", query)            # reuse §3's function

    @pipe.technique("answer", "small")
    async def small_model(ctx, query, doc_topic=None):
        return mock_model("small", "grounded", ctx.item["topic"], doc_topic, random.Random(0))

    @pipe.technique("answer", "large")
    async def large_model(ctx, query, doc_topic=None):
        return mock_model("large", "grounded", ctx.item["topic"], doc_topic, random.Random(0))

    @pipe.compose
    async def run(config, item, ctx):
        doc = await ctx.run("retrieval", query=item["text"])
        return await ctx.run("answer", query=item["text"], doc_topic=doc)

    class KeywordJudge:                              # same judge, CAFE's interface
        model = "keyword-match"
        async def score(self, rubric, question, answer, reference=None):
            v = keyword_judge(str(answer), reference or "")
            return cafe.JudgeOutput(value=v, value_numeric=v, reasoning="",
                                    prompt="", raw_response=str(v))

    study = cafe.Study(
        name="meridian-support-bot",
        system=pipe,
        factors=[pipe.factor("retrieval"), pipe.factor("answer")],
        dataset=[{"text": q, "reference": r, "topic": t} for t, q, r in GOLDEN],
        rubric=cafe.rubrics.CORRECTNESS_0_3,
        judge=KeywordJudge(),
        replications=6,
    )
    print(study.evaluate().report())    # means, significance, effect sizes — CLMM via R
else:
    print("Skipped (cafe-core not installed).")

For a real study you swap exactly two things: the technique bodies become real model calls (`cafe.complete(...)` or your `llm_providers.py` classes), and the judge becomes `cafe.LLMJudge(model=...)` — e.g. a keyless local model via `ollama_cloud/gpt-oss:20b`, or any provider you have keys for.

### When the grid explodes: fractional factorial designs

Six binary factors = 64 configs; with 10 questions × 6 replications that's 3,840 judged runs. A **fractional factorial** design runs a carefully-chosen *fraction* (e.g. half or quarter) such that all main effects — and the interactions you care about — remain estimable; what you give up is the ability to separate high-order interactions (which are almost always negligible). This is a solved problem in classical DoE, and CAFE generates these designs for you. The practical guidance:

- **≤4 factors** → just run the full factorial.
- **5–8 factors** → half- or quarter-fraction; keep full resolution on the 2–3 factors you most suspect interact.
- **Screening many knobs** → run a heavily fractionated design first to find the 2–3 factors that matter, then a full factorial on those.

### The paper

CAFE comes out of statistics + ML research (Lukassen, Weisser, Kneib & Silbersdorff, [arXiv:2607.10380](https://arxiv.org/abs/2607.10380)) — yes, the same Weisser who wrote this course, so complaints about the API go straight to the source. The [docs](https://fabian-lu.github.io/Cafe/) include runnable tutorial notebooks for RAG, routing, and agentic systems.

---

## 🧪 Practice exercises

Work these in order — each has a worked solution, but wrestle first.

### Exercise 1 — A fourth factor

Add `temperature ∈ {low, high}` to the study: pass it through `run_pipeline` into `mock_model`, and make high temperature *subtract 0.08* from `p_correct` (creative-but-wrong). Re-run the full study (now 16 configs) and the ANOVA. Does the design detect the effect you injected? How does its partial η² compare to `prompt_style`'s?

<details>
<summary>✅ <b>Solution</b></summary>

```python
def mock_model_t(size, style, item_topic, doc_topic, rng, temperature="low"):
    has_right_doc = doc_topic == item_topic
    if has_right_doc:
        p = {"small": 0.82, "large": 0.93}[size]
        if style == "grounded":
            p = min(p + 0.05, 0.98)
    else:
        p = {"small": 0.25, "large": 0.48}[size]
    if temperature == "high":
        p = max(p - 0.08, 0.02)
    roll = rng.random()
    if roll < p:
        return f"According to the docs: {KB[item_topic]}"
    if roll < p + 0.30:
        return "That's covered in the docs — check the relevant settings page."
    return f"According to the docs: {KB[rng.choice([t for t in KB if t != item_topic])]}"

factors4 = {**FACTORS, "temperature": ["low", "high"]}
rng4 = random.Random(1)
rows4 = []
for combo in itertools.product(*factors4.values()):
    cfg = dict(zip(factors4, combo))
    for qid, (topic, q, ref) in enumerate(GOLDEN):
        for rep in range(REPLICATIONS):
            doc = retrieve(cfg["retrieval"], q)
            ans = mock_model_t(cfg["model_size"], cfg["prompt_style"], topic, doc,
                               rng4, cfg["temperature"])
            rows4.append({**cfg, "qid": f"q{qid:02d}",
                          "score": keyword_judge(ans, ref)})
res4 = pd.DataFrame(rows4)
ols4 = smf.ols("score ~ C(retrieval) * C(model_size) + C(prompt_style)"
               " + C(temperature) + C(qid)", data=res4).fit()
an4 = sm.stats.anova_lm(ols4, typ=2)
an4["partial_eta_sq"] = an4["sum_sq"] / (an4["sum_sq"] + an4.loc["Residual", "sum_sq"])
print(an4.round(4))
```

A −0.08 shift in correctness probability is a *small* effect — with 960 runs the ANOVA usually flags it, but its partial η² sits in the same "small" band as `prompt_style`. The meta-lesson: the machinery recovers exactly what you injected, which is why simulating a study like this is the standard way to sanity-check an evaluation harness before spending real API money.
</details>

### Exercise 2 — 🐞 Debug me: the biased comparison

A colleague "simplifies" the analysis like this — and concludes retrieval matters far less than §6 claims:

```python
# 🐞 What's wrong here?
good_runs = results[results["score"] > 0]            # "drop the failed runs first"
effect = good_runs.groupby("retrieval")["score"].mean()
print(effect)   # the retrieval gap shrinks to less than half its §6 size!
```

The code runs without error and the numbers are real. Why is the conclusion garbage?

<details>
<summary>✅ <b>Solution</b></summary>

```python
# The bug is the filter, not the groupby: conditioning on the OUTCOME (score > 0)
# before comparing factors is selection bias. Retrieval's main effect is precisely
# that it converts 0-score runs into 3-score runs — throw away the zeros and
# you've thrown away the effect. Compare on ALL runs:
print(results.groupby("retrieval")["score"].mean())          # the honest gap
print(results.groupby("retrieval")["score"]
      .apply(lambda s: (s == 0).mean()).round(2))            # where the zeros live
```

This is the evaluation-flavoured version of survivorship bias ("filter to sessions that completed, then compare funnels"). Rule: never condition on a variable *downstream* of the treatment you're comparing.
</details>

### Exercise 3 — How many replications do I need?

Re-run the §5 study with `REPLICATIONS` in `{1, 2, 4, 8}` (fresh `random.Random(seed)` per run, same seed across settings) and record the ANOVA p-value of `C(model_size)` each time. Plot p-value vs replications. Where does the model-size effect become reliably detectable?

<details>
<summary>✅ <b>Solution</b></summary>

```python
pvals = {}
for n_rep in [1, 2, 4, 8]:
    r = random.Random(7)
    rws = []
    for cfg in configs:
        for qid, item in enumerate(GOLDEN):
            for rep in range(n_rep):
                out = run_pipeline(cfg, item, r)
                rws.append({**cfg, "qid": f"q{qid:02d}",
                            "score": keyword_judge(out["answer"], item[2])})
    d = pd.DataFrame(rws)
    a = sm.stats.anova_lm(
        smf.ols("score ~ C(retrieval) * C(model_size) + C(prompt_style) + C(qid)",
                data=d).fit(), typ=2)
    pvals[n_rep] = a.loc["C(model_size)", "PR(>F)"]
ax = pd.Series(pvals).plot(marker="o", logy=True)
ax.axhline(0.05, color="red", ls="--", label="α = 0.05")
ax.set_xlabel("replications"); ax.set_ylabel("p-value of model_size (log)")
ax.legend()
```

The big retrieval effect is significant almost immediately; the *smaller* model-size effect needs several replications before its p-value reliably clears 0.05. This is statistical **power**: replication count should be sized to the smallest effect you care about detecting, not to your patience.
</details>

## 🧠 Stretch exercises

No worked solutions — these are small projects. Sketches of the approach are included.

- **A. Hand-rolled half-fraction.** Take the 16-config grid from Exercise 1 and keep only the 8 configs where an *even number* of factors are at their "high" level (this parity rule is the classic 2⁴⁻¹ half-fraction). Re-run the ANOVA with main effects only. Do the main-effect estimates survive with half the runs?
- **B. A real LLM judge.** Replace `keyword_judge` with an LLM judge via [`llm_providers.py`](../llm_providers.py) using the NB 32 judge-prompt pattern (rubric in the system prompt, answer + reference in the user turn, force a single digit out). Then measure keyword-judge ↔ LLM-judge agreement on 50 shared answers.
- **C. Binary rubric, correct model.** Collapse the score to `pass = score == 3` and refit with `smf.logit` (add `C(qid)`). Compare the factor story to the OLS one — this is the "scale-matched model" idea from §6 with the tools you already have.
- **D. Evaluate a real pipeline.** Wrap the NB 29 embeddings-retrieval demo as `run(config, item)` with factors `retrieval ∈ {keyword, embedding}` × `k ∈ {1, 3}` and run this notebook's §5–§7 machinery on it unchanged. This is the moment the harness stops being a toy.

## 🎁 Bonus mini-project — a full CAFE study

Install `cafe-core` (+ R) and reproduce this notebook's study end-to-end in CAFE: the §8 skeleton, `replications=6`, then `study.evaluate().report()`. Compare CAFE's CLMM factor table against our OLS `anova` table — same ranking? Then swap in a real model via a `cafe.LLMJudge` and one real technique, and re-run. If you have Docker: `docker compose up` in `apps/web-app` and click through the same study in the browser ([live demo](https://cafe-ai.de/demo) if you'd rather not install).

---

## ✅ Self-assessment

Before moving on, you should be able to:

- [ ] Explain the **attribution problem** in compound AI systems and why a before/after benchmark score can't solve it
- [ ] Define **factor / level / configuration** and enumerate a full factorial design in one line of Python
- [ ] Explain why **replication** is non-negotiable for LLM evaluation, and demonstrate the single-run instability on data
- [ ] Fit an ANOVA with a blocking term and an interaction, and read **p-values and partial η²** as "is it real?" / "how big is it?"
- [ ] State why an **ordinal rubric** calls for a CLMM rather than OLS, and which tool does that for you
- [ ] Compute and read a **cost/quality Pareto frontier**, and use it to kill a dominated configuration
- [ ] Map every piece of the hand-rolled harness onto its **CAFE** counterpart

🧠 **Check what stuck:** [Module 18 quiz](../quizzes/quiz_18_compound_ai_evaluation.ipynb) — five questions, ~10 minutes.

🚀 **Next:** this is the course's last optional module — the natural next step is to point this machinery at your own work. Take your Module 15 capstone (or the NB 29–31 RAG stack) and run a real factorial study on it: that artefact — *"we measured which technique matters, here's the frontier"* — is the strongest piece an AI-engineering portfolio can hold.